# Deep Integration — Cross-Modal Autoencoder
**Person 2 (Bojana) — Run with T4 GPU**

This implements the third fusion strategy promised in the project proposal:
> *'Длабока интеграција — автоенкодер за учење на заедничка латентна репрезентација'*

Architecture:
- **Encoder**: Geneformer (1152-dim) + MethylGPT (128-dim) → shared latent space (64-dim)
- **Decoder**: reconstructs both modalities from the latent space
- The latent space captures cross-modal biological information
- Classify from latent space with XGBoost

**Runtime → Change runtime type → T4 GPU → Run all** (~20 min)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch numpy scikit-learn xgboost matplotlib umap-learn

In [ ]:
SHARED_FOLDER_NAME = "multiomics-project"
from pathlib import Path
DRIVE_ROOT   = Path(f"/content/drive/MyDrive/{SHARED_FOLDER_NAME}")
GF_EMB_PATH  = DRIVE_ROOT / "data/processed/geneformer_embeddings.npy"
MG_EMB_PATH  = DRIVE_ROOT / "data/processed/methylgpt_embeddings.npy"
MANIFEST     = DRIVE_ROOT / "data/manifests/matched_samples.csv"
OUTPUT_DIR   = DRIVE_ROOT / "data/processed"

LATENT_DIM   = 64    # shared latent space dimension
EPOCHS       = 200
BATCH_SIZE   = 64
LR           = 1e-3

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'GF embeddings exist : {GF_EMB_PATH.exists()}')
print(f'MG embeddings exist : {MG_EMB_PATH.exists()}')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

gf_emb   = np.load(GF_EMB_PATH).astype(np.float32)   # (800, 1152)
mg_emb   = np.load(MG_EMB_PATH).astype(np.float32)   # (800, 128)
manifest = pd.read_csv(MANIFEST)
labels   = manifest['project'].values
le       = LabelEncoder()
y        = le.fit_transform(labels)

# Standardize each modality
gf_scaler = StandardScaler()
mg_scaler = StandardScaler()
gf_norm   = gf_scaler.fit_transform(gf_emb)
mg_norm   = mg_scaler.fit_transform(mg_emb)

print(f'Geneformer : {gf_emb.shape}')
print(f'MethylGPT  : {mg_emb.shape}')
print(f'Classes    : {le.classes_}')

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CrossModalAutoencoder(nn.Module):
    """
    Learns a shared latent representation from two omics modalities.
    Encoder: each modality → shared latent space
    Decoder: latent space → reconstruct both modalities
    """
    def __init__(self, gf_dim=1152, mg_dim=128, latent_dim=64):
        super().__init__()
        # Modality-specific encoders
        self.gf_encoder = nn.Sequential(
            nn.Linear(gf_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),    nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),    nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.mg_encoder = nn.Sequential(
            nn.Linear(mg_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 128),    nn.BatchNorm1d(128), nn.ReLU(),
        )
        # Shared latent projection (fuse both modalities)
        self.shared_encoder = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        # Decoders — reconstruct each modality from latent
        self.gf_decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 512),        nn.ReLU(),
            nn.Linear(512, gf_dim)
        )
        self.mg_decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, mg_dim)
        )

    def encode(self, gf, mg):
        h_gf = self.gf_encoder(gf)
        h_mg = self.mg_encoder(mg)
        z    = self.shared_encoder(torch.cat([h_gf, h_mg], dim=1))
        return z

    def forward(self, gf, mg):
        z      = self.encode(gf, mg)
        gf_rec = self.gf_decoder(z)
        mg_rec = self.mg_decoder(z)
        return z, gf_rec, mg_rec

model = CrossModalAutoencoder(gf_dim=gf_emb.shape[1], mg_dim=mg_emb.shape[1], latent_dim=LATENT_DIM)
model.to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

gf_tensor = torch.tensor(gf_norm, dtype=torch.float32)
mg_tensor = torch.tensor(mg_norm, dtype=torch.float32)
dataset   = TensorDataset(gf_tensor, mg_tensor)
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'Training autoencoder for {EPOCHS} epochs...')
losses = []
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for gf_batch, mg_batch in loader:
        gf_batch = gf_batch.to(device)
        mg_batch = mg_batch.to(device)
        z, gf_rec, mg_rec = model(gf_batch, mg_batch)
        # Reconstruction loss for both modalities
        loss_gf = F.mse_loss(gf_rec, gf_batch)
        loss_mg = F.mse_loss(mg_rec, mg_batch)
        loss    = loss_gf + loss_mg
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    losses.append(epoch_loss / len(loader))
    if (epoch + 1) % 20 == 0:
        print(f'  Epoch {epoch+1:3d}/{EPOCHS}  Loss={losses[-1]:.4f}')

print('Training complete!')

In [ ]:
# Extract latent embeddings for all samples
model.eval()
with torch.no_grad():
    z_all = model.encode(
        gf_tensor.to(device),
        mg_tensor.to(device)
    ).cpu().numpy()

print(f'Autoencoder latent embedding shape: {z_all.shape}')  # (800, 64)
np.save(OUTPUT_DIR / 'autoencoder_embeddings.npy', z_all)
print(f'Saved to Drive!')

In [ ]:
# Classify with XGBoost — same setup as other experiments
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

def run_kfold(X, y, le, label):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accs, f1s, aucs = [], [], []
    for train_idx, val_idx in skf.split(X, y):
        clf = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8,
                                  eval_metric='mlogloss', random_state=42, n_jobs=-1)
        clf.fit(X[train_idx], y[train_idx])
        pred   = clf.predict(X[val_idx])
        proba  = clf.predict_proba(X[val_idx])
        accs.append(accuracy_score(y[val_idx], pred))
        f1s.append(f1_score(y[val_idx], pred, average='macro', zero_division=0))
        y_bin = label_binarize(y[val_idx], classes=list(range(len(le.classes_))))
        aucs.append(roc_auc_score(y_bin, proba, multi_class='ovr', average='macro'))
    print(f'{label}:  Acc={np.mean(accs):.3f}±{np.std(accs):.3f}  '
          f'F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}  '
          f'AUC={np.mean(aucs):.3f}±{np.std(aucs):.3f}')
    return np.mean(accs), np.mean(f1s), np.mean(aucs)

print('Results:')
acc, f1, auc_val = run_kfold(z_all, y, le, 'Deep Fusion (Autoencoder)')

# Save result
pd.DataFrame([{'Model':'Deep Fusion (Autoencoder)',
               'Accuracy': f'{acc:.3f}', 'F1': f'{f1:.3f}', 'AUC': f'{auc_val:.3f}'}]
).to_csv(OUTPUT_DIR / 'autoencoder_metrics.csv', index=False)
print('Metrics saved!')

In [ ]:
# UMAP of autoencoder latent space
import umap
import matplotlib.pyplot as plt

CANCER_COLORS = {'TCGA-BRCA':'#e41a1c','TCGA-LUAD':'#377eb8','TCGA-COAD':'#4daf4a',
                 'TCGA-KIRC':'#ff7f00','TCGA-LIHC':'#984ea3','TCGA-THCA':'#a65628'}

reducer   = umap.UMAP(n_components=2, random_state=42, n_neighbors=15)
embedding = reducer.fit_transform(z_all)

fig, ax = plt.subplots(figsize=(8, 7))
for cancer in np.unique(labels):
    mask = labels == cancer
    ax.scatter(embedding[mask,0], embedding[mask,1],
               c=CANCER_COLORS.get(cancer,'#333'), label=cancer.replace('TCGA-',''),
               s=30, alpha=0.85, edgecolors='none')
ax.set_title('UMAP — Deep Fusion (Autoencoder, 64-dim latent)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.set_aspect('equal','datalim')
plt.tight_layout()
plt.savefig(OUTPUT_DIR.parent / 'results/umap_autoencoder.png', dpi=150, bbox_inches='tight')
plt.show()
print('Done! autoencoder_embeddings.npy and metrics saved to Drive.')